# Batch Evaluation Subsampling

Arquivo: `batch_eval_subsampling.ipynb`.

Objetivo deste notebook: medir estabilidade das metricas ja calculaveis a partir dos `results_csv`, usando subamostragem estratificada por `semantic_intent` ouro.

Este notebook nao roda inferencia, nao ajusta thresholds e nao sobrescreve `evaluation/<dataset>/metrics/`. Ele le `evaluation/<dataset>/results_csv/<dataset>_predictions.csv`, reconstrui os mesmos `evaluation_examples` usados pelo `batch_eval_metrics.ipynb` e exporta uma nova arvore em `metrics_subsampling`.

Fluxo manual recomendado:

1. Rode bootstrap/imports.
2. Confira a configuracao geral.
3. Rode as funcoes auxiliares.
4. Rode Field Operations como analise principal.
5. Rode SNIPS, ATIS e MultiWOZ como analises complementares.
6. Use a celula final para visualizar os CSVs exportados.


In [ ]:
# Celula 1 - bootstrap/imports.
# Execute uma vez no inicio. Nao carrega modelos; apenas importa pandas e helpers de metricas.

import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'src').exists() and (candidate / 'datasets').exists():
            return candidate
    raise RuntimeError('Nao encontrei a raiz do projeto a partir do diretorio atual.')


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import display

from src.data_tools.batch_eval_utils import load_semantic_alias_registry
from src.data_tools.batch_metrics_utils import (
    compute_batch_metrics_reports,
    compute_intent_accuracy_reports,
    compute_slot_conll_reports,
    load_prediction_results_csvs,
)

print('REPO_ROOT =', REPO_ROOT)

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)


In [ ]:
# Celula 2 - configuracao geral da subamostragem.
# Ajuste apenas se quiser mudar seed, numero de rodadas ou tamanhos por intent.
# As predicoes e thresholds usados sao os que ja estao congelados nos results_csv.

REGISTRY_PATH = REPO_ROOT / 'datasets' / 'metadata' / 'semantic_alias_registry.json'
EVALUATION_ROOT = REPO_ROOT / 'evaluation'
SUBSAMPLING_DIRNAME = 'metrics_subsampling'
SUBSAMPLING_METHOD = 'stratified_by_intent'

SEED = 42
DEFAULT_ROUNDS = 1000
PREVIEW_ROWS = 5

# Analise principal: Field Operations.
# n=300 e tratado como ancora do dataset completo, com uma unica rodada.
FIELD_OPERATIONS_N_VALUES = [25, 50, 100, 150, 200, 250, 300]

# Analises complementares nos datasets publicos.
PUBLIC_DATASET_N_VALUES = {
    'snips': [25, 50, 100, 250],
    'atis': [25, 50],
    'multiwoz': [25, 50, 75],
}

DATASET_SAMPLING_PLAN = {
    'field_operations': FIELD_OPERATIONS_N_VALUES,
    **PUBLIC_DATASET_N_VALUES,
}

METRIC_COLUMNS = [
    'intent_accuracy',
    'intent_macro_precision',
    'intent_macro_recall',
    'intent_macro_f1',
    'intent_micro_f1',
    'slot_macro_precision_by_intent',
    'slot_macro_recall_by_intent',
    'slot_macro_f1_by_intent',
    'slot_micro_precision',
    'slot_micro_recall',
    'slot_micro_f1',
]

print('Subsampling method:', SUBSAMPLING_METHOD)
print('Default rounds:', DEFAULT_ROUNDS)
print('Seed:', SEED)
print('Sampling plan:', DATASET_SAMPLING_PLAN)


In [ ]:
# Celula 3 - carregamento dos results_csv e reconstrucao dos evaluation_examples.
# Execute antes das celulas por dataset. Nao roda inferencia; apenas usa os CSVs existentes.
# A reconstrucao usa compute_batch_metrics_reports, a mesma base do batch_eval_metrics.ipynb.

def get_dataset_predictions_csv(dataset_name: str) -> Path:
    return EVALUATION_ROOT / dataset_name / 'results_csv' / f'{dataset_name}_predictions.csv'


def get_subsampling_output_dir(
    dataset_name: str,
    *,
    rounds: int = DEFAULT_ROUNDS,
    seed: int = SEED,
    config_name: str | None = None,
) -> Path:
    config = config_name or f'r{rounds}_seed{seed}'
    return EVALUATION_ROOT / dataset_name / SUBSAMPLING_DIRNAME / SUBSAMPLING_METHOD / config


def load_dataset_for_subsampling(dataset_name: str):
    csv_path = get_dataset_predictions_csv(dataset_name)
    if not csv_path.exists():
        raise FileNotFoundError(f'Predictions CSV nao encontrado: {csv_path}')

    registry = load_semantic_alias_registry(REGISTRY_PATH)
    results_view = load_prediction_results_csvs([csv_path])
    if results_view.empty:
        raise ValueError(f'Predictions CSV sem linhas: {csv_path}')

    reports = compute_batch_metrics_reports(results_view, registry=registry)
    evaluation_examples = reports['evaluation_examples'].copy()
    if evaluation_examples.empty:
        raise ValueError(f'Sem evaluation_examples para {dataset_name}.')

    support = (
        evaluation_examples.groupby('semantic_intent', sort=True)
        .size()
        .rename('support')
        .reset_index()
    )

    print('=' * 80)
    print(f'Dataset carregado para subamostragem: {dataset_name}')
    print('Predictions CSV:', csv_path)
    print('Linhas:', len(evaluation_examples))
    print('Intents:', support.shape[0])
    print('Menor suporte:', int(support['support'].min()))
    display(support)

    return {
        'dataset_name': dataset_name,
        'csv_path': csv_path,
        'results_view': results_view,
        'reports': reports,
        'evaluation_examples': evaluation_examples,
        'support': support,
    }


In [ ]:
# Celula 4 - funcoes de amostragem, metricas por rodada e exportacao.
# Execute antes das celulas por dataset. Todas as metricas usam os mesmos helpers do batch_eval_metrics.
# A amostragem e feita por utterance completa, estratificada pelo semantic_intent ouro.

def build_sampling_coverage(evaluation_examples: pd.DataFrame, n_per_intent: int, rounds_to_run: int, seed: int):
    support = (
        evaluation_examples.groupby('semantic_intent', sort=True)
        .size()
        .rename('support')
        .reset_index()
    )
    if support.empty:
        raise ValueError('Nao ha semantic_intent para estratificar.')
    min_support = int(support['support'].min())
    if n_per_intent > min_support:
        raise ValueError(
            f'n_per_intent={n_per_intent} excede o menor suporte do dataset ({min_support}). '
            'Use um n menor para manter a amostra balanceada por intent.'
        )

    full_support_anchor = bool((support['support'] == n_per_intent).all())
    coverage = support.copy()
    coverage['n_per_intent'] = n_per_intent
    coverage['rounds_requested'] = rounds_to_run
    coverage['rounds_executed'] = 1 if full_support_anchor else rounds_to_run
    coverage['sampled_per_round'] = n_per_intent
    coverage['total_sampled_across_rounds'] = coverage['sampled_per_round'] * coverage['rounds_executed']
    coverage['seed'] = seed
    coverage['full_support_anchor'] = full_support_anchor
    return coverage, full_support_anchor


def sample_by_gold_intent(evaluation_examples: pd.DataFrame, n_per_intent: int, round_index: int, seed: int):
    sampled_parts = []
    for group_index, (_semantic_intent, group) in enumerate(evaluation_examples.groupby('semantic_intent', sort=True)):
        sampled_parts.append(
            group.sample(
                n=n_per_intent,
                replace=False,
                random_state=seed + round_index * 1009 + group_index * 9173,
            )
        )
    return pd.concat(sampled_parts, ignore_index=True)


def compute_round_metrics(
    sampled: pd.DataFrame,
    *,
    dataset_name: str,
    n_per_intent: int,
    round_number: int,
    rounds_requested: int,
    seed: int,
    full_support_anchor: bool,
):
    intent_summary, intent_by_label, _intent_confusion = compute_intent_accuracy_reports(sampled)
    slot_summary, slot_by_label, slot_by_intent = compute_slot_conll_reports(sampled)

    intent_row = intent_summary.iloc[0]
    slot_row = slot_summary.iloc[0]
    round_row = {
        'dataset_name': dataset_name,
        'n_per_intent': n_per_intent,
        'round': round_number,
        'rounds_requested': rounds_requested,
        'seed': seed,
        'rows': len(sampled),
        'intent_count': int(intent_row['intent_count']),
        'full_support_anchor': full_support_anchor,
        'intent_accuracy': float(intent_row['intent_accuracy']),
        'intent_macro_precision': float(intent_row['macro_precision']),
        'intent_macro_recall': float(intent_row['macro_recall']),
        'intent_macro_f1': float(intent_row['macro_f1']),
        'intent_micro_f1': float(intent_row['micro_f1']),
        'slot_macro_precision_by_intent': float(slot_row['precision']),
        'slot_macro_recall_by_intent': float(slot_row['recall']),
        'slot_macro_f1_by_intent': float(slot_row['f1']),
        'slot_micro_precision': float(slot_row['micro_precision']),
        'slot_micro_recall': float(slot_row['micro_recall']),
        'slot_micro_f1': float(slot_row['micro_f1']),
    }

    for frame in (intent_by_label, slot_by_intent, slot_by_label):
        frame.insert(0, 'full_support_anchor', full_support_anchor)
        frame.insert(0, 'seed', seed)
        frame.insert(0, 'rounds_requested', rounds_requested)
        frame.insert(0, 'round', round_number)
        frame.insert(0, 'n_per_intent', n_per_intent)
        frame.insert(0, 'dataset_name', dataset_name)

    return round_row, intent_by_label, slot_by_intent, slot_by_label


def summarize_rounds(rounds: pd.DataFrame) -> pd.DataFrame:
    summary_rows = []
    group_columns = ['dataset_name', 'n_per_intent']
    for (dataset_name, n_per_intent), group in rounds.groupby(group_columns, sort=True):
        for metric in METRIC_COLUMNS:
            values = group[metric].astype(float)
            summary_rows.append(
                {
                    'dataset_name': dataset_name,
                    'n_per_intent': int(n_per_intent),
                    'metric': metric,
                    'mean': float(values.mean()),
                    'std': float(values.std(ddof=1)) if len(values) > 1 else 0.0,
                    'min': float(values.min()),
                    'p2_5': float(values.quantile(0.025)),
                    'median': float(values.median()),
                    'p97_5': float(values.quantile(0.975)),
                    'max': float(values.max()),
                    'rounds': int(len(values)),
                    'rows_per_round': int(group['rows'].iloc[0]),
                    'rounds_requested': int(group['rounds_requested'].iloc[0]),
                    'seed': int(group['seed'].iloc[0]),
                    'full_support_anchor': bool(group['full_support_anchor'].iloc[0]),
                }
            )
    return pd.DataFrame(summary_rows)


def run_stratified_subsampling(
    dataset_name: str,
    n_values: list[int],
    *,
    rounds: int = DEFAULT_ROUNDS,
    seed: int = SEED,
    config_name: str | None = None,
):
    loaded = load_dataset_for_subsampling(dataset_name)
    evaluation_examples = loaded['evaluation_examples']
    output_dir = get_subsampling_output_dir(
        dataset_name,
        rounds=rounds,
        seed=seed,
        config_name=config_name,
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    coverage_frames = []
    round_rows = []
    intent_by_label_frames = []
    slot_by_intent_frames = []
    slot_by_label_frames = []

    for n_per_intent in n_values:
        coverage, full_support_anchor = build_sampling_coverage(
            evaluation_examples,
            n_per_intent=n_per_intent,
            rounds_to_run=rounds,
            seed=seed,
        )
        coverage_frames.append(coverage)
        rounds_executed = int(coverage['rounds_executed'].iloc[0])
        print(f'{dataset_name}: n={n_per_intent}, rounds={rounds_executed}, full_support_anchor={full_support_anchor}')

        for round_index in range(rounds_executed):
            round_number = round_index + 1
            sampled = sample_by_gold_intent(
                evaluation_examples,
                n_per_intent=n_per_intent,
                round_index=round_index,
                seed=seed,
            )
            round_row, intent_by_label, slot_by_intent, slot_by_label = compute_round_metrics(
                sampled,
                dataset_name=dataset_name,
                n_per_intent=n_per_intent,
                round_number=round_number,
                rounds_requested=rounds,
                seed=seed,
                full_support_anchor=full_support_anchor,
            )
            round_rows.append(round_row)
            intent_by_label_frames.append(intent_by_label)
            slot_by_intent_frames.append(slot_by_intent)
            slot_by_label_frames.append(slot_by_label)

    rounds_frame = pd.DataFrame(round_rows)
    summary = summarize_rounds(rounds_frame)
    coverage = pd.concat(coverage_frames, ignore_index=True)
    intent_by_label_rounds = pd.concat(intent_by_label_frames, ignore_index=True)
    slot_by_intent_rounds = pd.concat(slot_by_intent_frames, ignore_index=True)
    slot_by_label_rounds = pd.concat(slot_by_label_frames, ignore_index=True)

    written = {
        'summary': output_dir / 'summary.csv',
        'rounds': output_dir / 'rounds.csv',
        'coverage': output_dir / 'coverage.csv',
        'intent_by_label_rounds': output_dir / 'intent_by_label_rounds.csv',
        'slot_by_intent_rounds': output_dir / 'slot_by_intent_rounds.csv',
        'slot_by_label_rounds': output_dir / 'slot_by_label_rounds.csv',
    }
    summary.to_csv(written['summary'], index=False, encoding='utf-8-sig')
    rounds_frame.to_csv(written['rounds'], index=False, encoding='utf-8-sig')
    coverage.to_csv(written['coverage'], index=False, encoding='utf-8-sig')
    intent_by_label_rounds.to_csv(written['intent_by_label_rounds'], index=False, encoding='utf-8-sig')
    slot_by_intent_rounds.to_csv(written['slot_by_intent_rounds'], index=False, encoding='utf-8-sig')
    slot_by_label_rounds.to_csv(written['slot_by_label_rounds'], index=False, encoding='utf-8-sig')

    print('=' * 80)
    print(f'Subsampling concluido: {dataset_name}')
    print('Output dir:', output_dir)
    print('Summary')
    display(summary.head(PREVIEW_ROWS * 3))
    print('Coverage')
    display(coverage)
    print('Arquivos exportados:')
    for name, path in written.items():
        print(f'- {name}: {path}')

    return {
        'dataset_name': dataset_name,
        'output_dir': output_dir,
        'summary': summary,
        'rounds': rounds_frame,
        'coverage': coverage,
        'intent_by_label_rounds': intent_by_label_rounds,
        'slot_by_intent_rounds': slot_by_intent_rounds,
        'slot_by_label_rounds': slot_by_label_rounds,
        'written': written,
        'loaded': loaded,
    }


def run_configured_subsampling(dataset_name: str):
    if dataset_name not in DATASET_SAMPLING_PLAN:
        raise KeyError(f'Dataset sem plano configurado: {dataset_name}')
    return run_stratified_subsampling(
        dataset_name,
        n_values=DATASET_SAMPLING_PLAN[dataset_name],
        rounds=DEFAULT_ROUNDS,
        seed=SEED,
    )


## Field Operations - Analise Principal

Rode esta celula para a analise principal de estabilidade no dominio proprio. Ela le `evaluation/field_operations/results_csv/field_operations_predictions.csv` e exporta os CSVs em `evaluation/field_operations/metrics_subsampling/stratified_by_intent/r1000_seed42/`.


In [ ]:
# Celula Field Operations - analise principal.
# Pode demorar, pois roda 1000 amostras para cada n exceto a ancora do dataset completo.
# Nao roda inferencia e nao altera evaluation/field_operations/metrics/.

field_operations_subsampling = run_configured_subsampling('field_operations')


## SNIPS - Analise Complementar

Rode esta celula para avaliar estabilidade no benchmark SNIPS. Ela le `evaluation/snips/results_csv/snips_predictions.csv` e exporta em `evaluation/snips/metrics_subsampling/stratified_by_intent/r1000_seed42/`.


In [ ]:
# Celula SNIPS - analise complementar.
# Usa n por intent = 25, 50, 100, 250.
# Nao roda inferencia e nao altera evaluation/snips/metrics/.

snips_subsampling = run_configured_subsampling('snips')


## ATIS - Analise Complementar

Rode esta celula para avaliar estabilidade no ATIS filtrado. Ela le `evaluation/atis/results_csv/atis_predictions.csv` e exporta em `evaluation/atis/metrics_subsampling/stratified_by_intent/r1000_seed42/`.


In [ ]:
# Celula ATIS - analise complementar.
# Usa n por intent = 25, 50, respeitando o menor suporte do ATIS filtrado.
# Nao roda inferencia e nao altera evaluation/atis/metrics/.

atis_subsampling = run_configured_subsampling('atis')


## MultiWOZ - Analise Complementar

Rode esta celula para avaliar estabilidade no MultiWOZ filtrado. Ela le `evaluation/multiwoz/results_csv/multiwoz_predictions.csv` e exporta em `evaluation/multiwoz/metrics_subsampling/stratified_by_intent/r1000_seed42/`.


In [ ]:
# Celula MultiWOZ - analise complementar.
# Usa n por intent = 25, 50, 75, porque o menor suporte atual e 75.
# Nao roda inferencia e nao altera evaluation/multiwoz/metrics/.

multiwoz_subsampling = run_configured_subsampling('multiwoz')


## Preview Dos Arquivos Exportados

Rode esta celula depois de executar um ou mais datasets. Ela apenas procura os CSVs ja salvos em `metrics_subsampling` e mostra uma previa compacta.


In [ ]:
# Celula final - leitura/preview dos CSVs exportados.
# Use para conferir rapidamente quais analises ja foram geradas e olhar as primeiras linhas.

def preview_subsampling_outputs(dataset_names=None, *, rounds: int = DEFAULT_ROUNDS, seed: int = SEED):
    dataset_names = dataset_names or list(DATASET_SAMPLING_PLAN.keys())
    rows = []
    for dataset_name in dataset_names:
        output_dir = get_subsampling_output_dir(dataset_name, rounds=rounds, seed=seed)
        for filename in [
            'summary.csv',
            'rounds.csv',
            'coverage.csv',
            'intent_by_label_rounds.csv',
            'slot_by_intent_rounds.csv',
            'slot_by_label_rounds.csv',
        ]:
            path = output_dir / filename
            rows.append(
                {
                    'dataset_name': dataset_name,
                    'file': filename,
                    'exists': path.exists(),
                    'path': str(path),
                    'size_bytes': path.stat().st_size if path.exists() else 0,
                }
            )
    artifacts = pd.DataFrame(rows)
    display(artifacts)

    for dataset_name in dataset_names:
        summary_path = get_subsampling_output_dir(dataset_name, rounds=rounds, seed=seed) / 'summary.csv'
        if summary_path.exists():
            print('=' * 80)
            print(dataset_name, summary_path)
            display(pd.read_csv(summary_path).head(PREVIEW_ROWS * 3))

    return artifacts


subsampling_artifacts = preview_subsampling_outputs()
